# Socratic AI Tutor — SmolLM2-360M Fine-Tuning Pipeline

## Model: SmolLM2-360M-Instruct → Q4_K_M GGUF (~230 MB)

**Goal:** Fine-tune a 360M model that handles three behaviors:
1. **Socratic tutoring** — guides with questions (DS/ML concepts)
2. **Code assistance** — guides students to write code themselves (Socratic questioning)
3. **Casual conversation** — handles greetings and chitchat naturally

**Why SmolLM2-360M over Qwen3-0.6B:**

| | Qwen3-0.6B | SmolLM2-360M |
|---|---|---|
| Effective params | 751M (large vocab) | 360M |
| Vocab size | 151,936 tokens | 49,152 tokens |
| Q4_K_M size | ~461 MB | **~230 MB** |
| Galaxy A14 (4 GB) | OOM crash | **Fits comfortably** |
| Think blocks | Yes | No (simpler, faster) |
| Chat format | ChatML | ChatML ← same dataset! |

**No `<think>` blocks** — SmolLM2 generates direct responses. Socratic behavior is
enforced by the system prompt and fine-tuning, not chain-of-thought reasoning.
The smaller vocabulary means 2× smaller GGUF with the same parameter count.

In [1]:
%pip install -q torch datasets transformers peft bitsandbytes scikit-learn huggingface_hub accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 50.4 MB/s eta 0:00:00


In [2]:
# ============================================================================
# Cell 1: Imports, Seed, and System Prompt
# ============================================================================

import json, random, re, gc, os
from typing import Dict, List, Any, Tuple
from copy import deepcopy

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from dataclasses import dataclass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ============================================================================
# UNIFIED SYSTEM PROMPT
# This single prompt teaches the model three behaviors:
# 1. Socratic tutoring (conceptual questions)
# 2. Direct code help (implementation requests)
# 3. Casual conversation (greetings, chitchat)
#
# The model learns to detect intent from the user message and route to the
# correct behavior. This matches the production system prompt.
# ============================================================================
SYSTEM_PROMPT = """You are a Socratic AI tutor specializing in data science and machine learning.
RULES:
1. For conceptual questions: respond with ONE guiding question. NEVER give direct answers.
2. For code questions: guide the student to write code themselves through Socratic questioning.
3. For casual messages (greetings, thanks): respond warmly and naturally.
Never explain directly. Always guide through questions."""
print("\u2713 Environment configured")
print(f"  - PyTorch: {torch.__version__}")
print(f"  - CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


✓ Environment configured
  - PyTorch: 2.10.0+cu128
  - CUDA: True
  - GPU: NVIDIA L4
  - VRAM: 23.7 GB


In [3]:
from huggingface_hub import login
login()
print("\u2713 Authenticated")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✓ Authenticated


In [4]:
# ============================================================================
# Cell 2: Load Model + Tokenizer
# ============================================================================

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)
model.config.use_cache = False

print(f"\u2713 Model loaded: {model.num_parameters():,} params")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✓ Model loaded: 361,821,120 params


In [5]:
# ============================================================================
# Cell 3: Load and Merge Training Data
# ============================================================================
# We merge the original Socratic data with supplementary data covering:
# - Code Q&A examples (direct code help)
# - Casual greetings/chitchat
# - Additional diverse Socratic conversations
# ============================================================================

SOCRATIC_PATH = "multi_socratic_data.json"
SUPPLEMENTARY_PATH = "supplementary_data.json"

with open(SOCRATIC_PATH, 'r') as f:
    socratic_data = json.load(f)

with open(SUPPLEMENTARY_PATH, 'r') as f:
    supplementary_data = json.load(f)

print(f"Original Socratic conversations: {len(socratic_data)}")
print(f"Supplementary conversations: {len(supplementary_data)}")

# Count categories in supplementary
cats = {}
for s in supplementary_data:
    c = s.get('category', 'unknown')
    cats[c] = cats.get(c, 0) + 1
print(f"  Breakdown: {cats}")

# Strip the 'category' field before merging (not needed for training)
clean_supplementary = []
for conv in supplementary_data:
    clean_supplementary.append({"messages": conv["messages"]})

# Merge
all_data = socratic_data + clean_supplementary
print(f"\nTotal merged conversations: {len(all_data)}")

Original Socratic conversations: 234
Supplementary conversations: 73
  Breakdown: {'greeting': 17, 'code': 34, 'socratic': 22}

Total merged conversations: 307


In [ ]:
# ============================================================================
# Cell 4: Data Validation and Normalization
# ============================================================================
# SmolLM2 does NOT use <think> blocks. Strip any that exist in the dataset
# (the original Socratic data was prepared for Qwen3 and contains them).
# After stripping, assistant responses are clean direct text.
# ============================================================================

def validate_and_fix_conversation(conv: Dict, idx: int) -> Dict:
    """Strip <think>...</think> blocks and ensure proper message structure."""
    messages = conv["messages"]
    fixed = []
    stripped = 0

    for msg in messages:
        if msg["role"] == "assistant":
            content = msg["content"]
            # Remove <think>...</think> blocks (Qwen3-era data)
            if "<think>" in content:
                content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
                stripped += 1
            if content:
                fixed.append({"role": "assistant", "content": content})
        else:
            fixed.append(msg)

    if stripped > 0:
        print(f"  Conv {idx}: Stripped {stripped} think block(s)")

    return {"messages": fixed}

# Validate all conversations
print("Validating training data (stripping Qwen3-era <think> blocks)...")
validated_data = []
total_assistant_msgs = 0
total_stripped = 0

for i, conv in enumerate(all_data):
    n_assistant = sum(1 for m in conv["messages"] if m["role"] == "assistant")
    n_with_think = sum(1 for m in conv["messages"]
                       if m["role"] == "assistant" and "<think>" in m["content"])
    total_assistant_msgs += n_assistant
    total_stripped += n_with_think
    validated_data.append(validate_and_fix_conversation(conv, i))

print(f"\n✓ Validation complete")
print(f"  Total assistant messages: {total_assistant_msgs}")
print(f"  Stripped think blocks: {total_stripped}")
print(f"  Clean responses: {total_assistant_msgs}/{total_assistant_msgs} (100%)")

In [ ]:
# ============================================================================
# Cell 5: Data Augmentation via Conversation Windowing
# ============================================================================
# Each multi-turn conversation generates multiple training samples by taking
# progressively longer prefixes. This multiplies data ~3-4x without
# synthetic generation.
# ============================================================================

def augment_by_windowing(conversations: List[Dict]) -> List[Dict]:
    """Generate all valid sub-conversations ending on an assistant turn."""
    augmented = []
    for conv in conversations:
        messages = conv["messages"]

        # Separate system prompt
        if messages[0]["role"] == "system":
            dialogue = messages[1:]
        else:
            dialogue = messages

        # Generate windows ending on assistant turns
        for end_idx in range(2, len(dialogue) + 1, 2):
            window = dialogue[:end_idx]
            if window and window[-1]["role"] == "assistant":
                augmented.append({"messages": window})

    return augmented

augmented_data = augment_by_windowing(validated_data)
random.shuffle(augmented_data)

print(f"After windowing: {len(augmented_data)} samples")
print(f"Multiplier: {len(augmented_data) / len(validated_data):.1f}x")

In [ ]:
# ============================================================================
# Cell 6: Tokenization with Label Masking
# ============================================================================
# FIX: SmolLM2's tokenizer may encode <|im_start|> differently than Qwen3.
# convert_tokens_to_ids() can return UNK if the internal string key differs.
# Instead, we use tokenizer.encode() to get the *actual* token IDs used in
# the output, then pattern-match against those sequences.
# ============================================================================

MAX_LENGTH = 768

def format_conversation(messages: List[Dict]) -> List[Dict]:
    """Prepend the unified system prompt to every conversation."""
    non_system = [m for m in messages if m["role"] != "system"]
    return [{"role": "system", "content": SYSTEM_PROMPT}] + non_system


# ── Debug: verify SmolLM2 special token encoding ───────────────────────────
_header_ids = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
_end_ids    = tokenizer.encode("<|im_end|>", add_special_tokens=False)
print(f"Assistant header tokens : {_header_ids}  (len={len(_header_ids)})")
print(f"End tag tokens          : {_end_ids}  (len={len(_end_ids)})")
assert len(_header_ids) > 0, "Cannot encode <|im_start|>assistant — check model name"
assert len(_end_ids)    > 0, "Cannot encode <|im_end|> — check model name"
# ───────────────────────────────────────────────────────────────────────────


def find_assistant_spans(input_ids: List[int]) -> List[tuple]:
    """
    Find token spans of assistant responses for label masking.

    Pattern-matches the tokenized form of '<|im_start|>assistant\\n' instead
    of looking up individual token IDs — robust across tokenizer backends
    (Llama, Qwen, SmolLM2, etc.) where the same text may map to different IDs.
    """
    spans = []
    assistant_header = tokenizer.encode(
        "<|im_start|>assistant\n", add_special_tokens=False
    )
    end_tag   = tokenizer.encode("<|im_end|>", add_special_tokens=False)
    hlen      = len(assistant_header)
    elen      = len(end_tag)

    i = 0
    while i <= len(input_ids) - hlen:
        if input_ids[i : i + hlen] == assistant_header:
            start_idx = i + hlen
            end_idx   = len(input_ids)   # fallback if truncated
            for j in range(start_idx, len(input_ids) - elen + 1):
                if input_ids[j : j + elen] == end_tag:
                    end_idx = j + elen   # include the <|im_end|> token(s)
                    break
            if start_idx < end_idx:
                spans.append((start_idx, end_idx))
            i = end_idx
        else:
            i += 1
    return spans


def tokenize_with_labels(examples: Dict) -> Dict:
    all_input_ids, all_attention_masks, all_labels = [], [], []

    for messages in examples["messages"]:
        formatted = format_conversation(messages)
        text = tokenizer.apply_chat_template(
            formatted, tokenize=False, add_generation_prompt=False
        )
        tok = tokenizer(
            text, truncation=True, max_length=MAX_LENGTH,
            padding="max_length", return_tensors=None
        )
        input_ids = tok["input_ids"]
        labels = [-100] * len(input_ids)
        for start, end in find_assistant_spans(input_ids):
            for idx in range(start, min(end, len(labels))):
                labels[idx] = input_ids[idx]

        all_input_ids.append(input_ids)
        all_attention_masks.append(tok["attention_mask"])
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels
    }


dataset = Dataset.from_list(augmented_data)
tokenized = dataset.map(
    tokenize_with_labels, batched=True, batch_size=50,
    remove_columns=dataset.column_names, desc="Tokenizing"
)
split = tokenized.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"\n✓ Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")

# Verify label masking on first sample
sample = train_dataset[0]
n_trainable = sum(1 for t in sample['labels'] if t != -100)
n_total     = sum(1 for t in sample['attention_mask'] if t == 1)
print(f"Label check: {n_trainable}/{n_total} real tokens are trainable")

if n_trainable == 0:
    # Extra debug: show what the first sample actually looks like
    ids = sample['input_ids']
    print(f"\nDEBUG — first 50 token IDs: {ids[:50]}")
    print(f"DEBUG — decoded first 200 chars: {tokenizer.decode(ids[:50])}")
    print(f"DEBUG — assistant_header={_header_ids}, end_tag={_end_ids}")

assert n_trainable > 0, "No trainable tokens — see DEBUG output above"
print("✓ Label masking OK")

In [ ]:
# ============================================================================
# Cell 7: LoRA Configuration
# ============================================================================
# r=32 with RSLoRA gives good capacity for behavioral fine-tuning.
# Targeting all attention + MLP projections maximizes expressiveness.
# ============================================================================

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    use_rslora=True,         # Rank-Stabilized LoRA — effective scale = alpha/sqrt(r)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ============================================================================
# Cell 8: Training Configuration
# ============================================================================
# Key decisions:
# - lr=2e-4: Standard for LoRA (2e-5 was too low in previous version)
# - Effective batch 16: Stable gradients
# - 3 epochs: Enough with ~1000 augmented samples, avoids overfit
# - Early stopping patience=5: Don't stop too early
# - weight_decay=0.01: Light regularization for LoRA
# ============================================================================

torch.cuda.empty_cache()
gc.collect()

BATCH_SIZE = 4
GRAD_ACCUM = 4
EPOCHS = 4
LR = 2e-4

total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = max(20, int(total_steps * 0.06))

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

training_args = TrainingArguments(
    output_dir="./qwen-socratic-lora-v3",

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=EPOCHS,

    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,

    bf16=True,
    optim="adamw_torch_fused",
    weight_decay=0.01,
    max_grad_norm=1.0,

    gradient_checkpointing=True,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    dataloader_pin_memory=True,
)


@dataclass
class SocraticDataCollator:
    tokenizer: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        return {
            "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
            "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
            "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
        }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=SocraticDataCollator(tokenizer=tokenizer),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5, early_stopping_threshold=0.005)
    ]
)

print("\u2713 Trainer initialized")

In [ ]:
# ============================================================================
# Cell 9: Training
# ============================================================================

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

train_result = trainer.train()

print(f"\n✓ Training complete")
print(f"  Runtime:     {train_result.metrics['train_runtime']:.1f}s")
print(f"  Final loss:  {train_result.metrics['train_loss']:.4f}")
print(f"  Steps:       {train_result.global_step}")

ADAPTER_DIR = "./smollm2-socratic-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✓ Adapter saved to {ADAPTER_DIR}")

In [ ]:
# ============================================================================
# Cell 10: Merge LoRA into Base Model
# ============================================================================

merged_model = model.merge_and_unload()

MERGED_DIR = "./smollm2-socratic-merged"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"✓ Merged model saved to {MERGED_DIR}")

In [ ]:
# ============================================================================
# Cell 11: Comprehensive Evaluation
# ============================================================================
# SmolLM2 generates direct responses — no <think> blocks to strip or check.
# Metrics focus on:
#   Socratic: ends with question, no direct answers, concise
#   Code:     guides with question, no raw code dumps
#   Casual:   warm tone, not overly Socratic
# ============================================================================

def generate_response(question: str, model_to_use, max_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_use.device)
    model_to_use.config.use_cache = True

    with torch.no_grad():
        outputs = model_to_use.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    model_to_use.config.use_cache = False
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    return response.strip()


def evaluate_socratic(response: str) -> Dict[str, bool]:
    return {
        "Ends with Question": response.rstrip().endswith("?"),
        "No Direct Answer": not any(p in response.lower() for p in [
            "the answer is", "here's how", "you should", "the solution is"
        ]),
        "Concise": len(response.split()) < 80,
    }


def evaluate_code(response: str) -> Dict[str, bool]:
    """Code questions — model should guide with Socratic questions, not dump code."""
    return {
        "Guides with Question": response.rstrip().endswith("?"),
        "No Direct Code Dump": "```" not in response,
    }


def evaluate_casual(response: str) -> Dict[str, bool]:
    return {
        "Warm Tone": any(w in response.lower() for w in [
            "hello", "hey", "hi", "welcome", "great", "good", "glad",
            "help", "ready", "here", "thanks", "thank", "nice", "awesome",
            "happy", "sure", "absolutely", "of course", "!"
        ]),
        "Not Overly Socratic": not response.rstrip().endswith("?") or len(response.split()) < 20,
    }


# Test questions by category
socratic_questions = [
    "What is overfitting and how do I prevent it?",
    "Can you explain what gradient descent does?",
    "Why should I normalize my features before training?",
    "What's the difference between classification and regression?",
    "What does a confusion matrix tell me?",
    "What is regularization in machine learning?",
    "Explain the bias-variance tradeoff",
]

code_questions = [
    "How do I read a CSV file in pandas?",
    "Show me how to train a random forest in sklearn",
    "Write code to split data into train and test sets",
    "How do I create a scatter plot with matplotlib?",
    "Show me how to do one-hot encoding",
]

casual_questions = [
    "Hi",
    "How are you?",
    "Thanks for the help!",
    "Who are you?",
    "Bye!",
]

print("=" * 70)
print("COMPREHENSIVE EVALUATION")
print("=" * 70)

# Socratic evaluation
print("\n--- SOCRATIC MODE ---")
socratic_metrics = []
for i, q in enumerate(socratic_questions, 1):
    response = generate_response(q, merged_model)
    quality = evaluate_socratic(response)
    socratic_metrics.append(quality)
    passed = all(quality.values())
    print(f"\n[S{i}] Q: {q}")
    print(f"     R: {response[:200]}{'...' if len(response) > 200 else ''}")
    print(f"     {'PASS' if passed else 'FAIL'} | {quality}")

# Code evaluation
print("\n--- CODE MODE (Socratic guidance) ---")
code_metrics = []
for i, q in enumerate(code_questions, 1):
    response = generate_response(q, merged_model, max_tokens=300)
    quality = evaluate_code(response)
    code_metrics.append(quality)
    passed = all(quality.values())
    print(f"\n[C{i}] Q: {q}")
    print(f"     R: {response[:200]}{'...' if len(response) > 200 else ''}")
    print(f"     {'PASS' if passed else 'FAIL'} | {quality}")

# Casual evaluation
print("\n--- CASUAL MODE ---")
casual_metrics = []
for i, q in enumerate(casual_questions, 1):
    response = generate_response(q, merged_model)
    quality = evaluate_casual(response)
    casual_metrics.append(quality)
    passed = all(quality.values())
    print(f"\n[G{i}] Q: {q}")
    print(f"     R: {response[:200]}{'...' if len(response) > 200 else ''}")
    print(f"     {'PASS' if passed else 'FAIL'} | {quality}")

# Summary
import pandas as pd

print("\n" + "=" * 70)
print("METRICS SUMMARY")
print("=" * 70)

df_s = pd.DataFrame(socratic_metrics)
df_c = pd.DataFrame(code_metrics)
df_g = pd.DataFrame(casual_metrics)

s_comp = df_s.all(axis=1).mean() * 100
c_comp = df_c.all(axis=1).mean() * 100
g_comp = df_g.all(axis=1).mean() * 100

print(f"\nSocratic compliance: {s_comp:.0f}%")
print((df_s.mean() * 100).round(1).to_string())
print(f"\nCode compliance: {c_comp:.0f}%")
print((df_c.mean() * 100).round(1).to_string())
print(f"\nCasual compliance: {g_comp:.0f}%")
print((df_g.mean() * 100).round(1).to_string())

overall = (s_comp + c_comp + g_comp) / 3
print(f"\n>>> OVERALL COMPLIANCE: {overall:.0f}% <<<")

# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (df, title, comp) in zip(axes, [
    (df_s, "Socratic", s_comp),
    (df_c, "Code", c_comp),
    (df_g, "Casual", g_comp),
]):
    pct = df.mean() * 100
    colors = ['#2ecc71' if v >= 80 else '#e74c3c' for v in pct.values]
    ax.barh(pct.index, pct.values, color=colors)
    ax.set_xlim(0, 115)
    ax.set_title(f"{title} — {comp:.0f}%", fontsize=12, fontweight='bold')
    for i, v in enumerate(pct.values):
        ax.text(v + 2, i, f"{v:.0f}%", va='center', fontweight='bold')

plt.suptitle(f"SmolLM2-360M — Overall Compliance: {overall:.0f}%", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('smollm2_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to smollm2_evaluation.png")

In [ ]:
# ============================================================================
# Cell 12: Interactive Testing
# ============================================================================
# SmolLM2 outputs plain text — no think blocks to strip.
# ============================================================================

test_cases = [
    ("Socratic", "What is gradient descent?"),
    ("Code",     "Show me how to read a CSV file in pandas"),
    ("Casual",   "Hey! How are you doing?"),
    ("Socratic", "What is a neural network?"),
    ("Code",     "Write a function to normalize data"),
    ("Casual",   "Thanks, you've been really helpful!"),
]

print("=" * 70)
print("INTERACTIVE TEST")
print("=" * 70)

for mode, question in test_cases:
    response = generate_response(question, merged_model, max_tokens=300)
    ends_q = response.rstrip().endswith("?")
    print(f"\n[{mode}] {question}")
    print(f"  Ends with '?': {'Yes' if ends_q else 'No'}")
    print(f"  Response: {response[:300]}")
    print("-" * 70)

In [ ]:
# ============================================================================
# Cell 13: GGUF Quantization (Q4_K_M)
# ============================================================================
# Converts the merged safetensors model to GGUF Q4_K_M format.
# SmolLM2-360M expected output: ~230 MB (vs 461 MB for Qwen3-0.6B)
# No tokenizer_config.json patch needed — SmolLM2 doesn't have the
# extra_special_tokens format bug that Qwen3 had.
# ============================================================================

import os

# Install llama.cpp tools
!pip install -q llama-cpp-python

# Clone llama.cpp for the conversion script
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /tmp/llama_cpp 2>/dev/null || true
!pip install -q -r /tmp/llama_cpp/requirements/requirements-convert_hf_to_gguf.txt 2>/dev/null || true

MERGED_DIR  = "./smollm2-socratic-merged"
GGUF_FP16   = "./smollm2-socratic-f16.gguf"
GGUF_Q4     = "./smollm2-socratic-Q4_K_M.gguf"

# Step 1: Convert to GGUF FP16
print("Step 1: Converting to GGUF FP16...")
!python /tmp/llama_cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_FP16} --outtype f16

if os.path.exists(GGUF_FP16):
    size_mb = os.path.getsize(GGUF_FP16) / 1024 / 1024
    print(f"✓ FP16 GGUF: {size_mb:.1f} MB")
else:
    print("ERROR: FP16 GGUF not created!")

In [ ]:
# ============================================================================
# Cell 14: Quantize to Q4_K_M
# ============================================================================

# Build the quantize tool
!cd /tmp/llama_cpp && mkdir -p build && cd build && cmake .. -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1 && cmake --build . --target llama-quantize -j$(nproc) > /dev/null 2>&1

QUANTIZE_BIN = "/tmp/llama_cpp/build/bin/llama-quantize"
if not os.path.exists(QUANTIZE_BIN):
    QUANTIZE_BIN = "/tmp/llama_cpp/build/llama-quantize"

if os.path.exists(QUANTIZE_BIN):
    print("Step 2: Quantizing to Q4_K_M...")
    !{QUANTIZE_BIN} {GGUF_FP16} {GGUF_Q4} Q4_K_M

    if os.path.exists(GGUF_Q4):
        size_mb = os.path.getsize(GGUF_Q4) / 1024 / 1024
        print(f"\n✓ Q4_K_M GGUF: {size_mb:.1f} MB")
        print(f"  Target was ~230 MB — fits comfortably on Galaxy A14 (4 GB RAM)")
    else:
        print("ERROR: Quantized GGUF not created!")
else:
    print("WARNING: llama-quantize not found. Build manually:")
    print("  cd /tmp/llama_cpp && make llama-quantize")
    print(f"  Then run: llama-quantize {GGUF_FP16} {GGUF_Q4} Q4_K_M")

In [ ]:
# ============================================================================
# Cell 15: Save to Google Drive (optional, for Colab)
# ============================================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_PATH = "/content/drive/MyDrive/smollm2-socratic"
    os.makedirs(DRIVE_PATH, exist_ok=True)

    import shutil
    merged_drive = os.path.join(DRIVE_PATH, "merged")
    if os.path.exists(merged_drive):
        shutil.rmtree(merged_drive)
    shutil.copytree(MERGED_DIR, merged_drive)
    print(f"✓ Merged model saved to {merged_drive}")

    for gguf_file, name in [(GGUF_FP16, "fp16"), (GGUF_Q4, "q4_k_m")]:
        if os.path.exists(gguf_file):
            dest = os.path.join(DRIVE_PATH, os.path.basename(gguf_file))
            shutil.copy2(gguf_file, dest)
            size_mb = os.path.getsize(dest) / 1024 / 1024
            print(f"✓ {name} GGUF saved: {dest} ({size_mb:.1f} MB)")

    print(f"\nAll files saved to: {DRIVE_PATH}")

except ImportError:
    print("Not running in Colab — skipping Google Drive save.")
    print(f"Files are in: {MERGED_DIR}, {GGUF_FP16}, {GGUF_Q4}")

In [ ]:
# ============================================================================
# Cell 16: Upload to HuggingFace Hub (optional)
# ============================================================================

REPO_ID = "Omar-keita/DSML-Socratic-smollm2-360M"  # Change to your repo
UPLOAD  = False  # Set to True when ready to upload

if UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi()

    # Create repo if it doesn't exist
    api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

    if os.path.exists(GGUF_Q4):
        print(f"Uploading Q4_K_M GGUF to {REPO_ID}...")
        api.upload_file(
            path_or_fileobj=GGUF_Q4,
            path_in_repo="smollm2-socratic-360M-Q4_K_M.gguf",
            repo_id=REPO_ID,
            repo_type="model",
        )
        print(f"✓ Uploaded to https://huggingface.co/{REPO_ID}")
        print(f"\nUpdate Flutter app download URL in:")
        print(f"  socratic_app/lib/screens/model_setup_screen.dart")
        print(f"  → downloadUrl = 'https://huggingface.co/{REPO_ID}/resolve/main/smollm2-socratic-360M-Q4_K_M.gguf'")
    else:
        print("No GGUF file found. Run quantization first.")
else:
    print("Upload disabled. Set UPLOAD = True to push to HuggingFace.")

## Summary

### What this notebook does:
1. **Loads** 307 conversations (234 original Socratic + 73 supplementary: code, greetings, diverse)
2. **Strips** all `<think>` blocks from the Qwen3-era data (SmolLM2 doesn't use thinking mode)
3. **Augments** via conversation windowing (~3-4x multiplier → ~1,000 training samples)
4. **Fine-tunes** SmolLM2-360M-Instruct with LoRA (r=32, RSLoRA) for 4 epochs
5. **Evaluates** all three modes: Socratic, Code, and Casual
6. **Quantizes** to Q4_K_M GGUF (~230 MB) for mobile deployment
7. **Saves** to Google Drive and optionally uploads to HuggingFace

### After running:
- Upload the Q4_K_M GGUF to HuggingFace
- Update `downloadUrl` in `socratic_app/lib/screens/model_setup_screen.dart`
- The ~230 MB model fits comfortably on budget phones including Galaxy A14 (4 GB RAM)

### Key differences from Qwen3-0.6B notebook:
| Aspect | Qwen3-0.6B | SmolLM2-360M |
|--------|-----------|--------------|
| Think blocks | Required in training data | Stripped out |
| `enable_thinking` | Yes | Not applicable |
| tokenizer_config patch | Needed (extra_special_tokens bug) | Not needed |
| GGUF size | ~461 MB | ~230 MB |
| Evaluation metrics | Has Think Block + others | Question/tone/directness only |